# Kural — Tamil Speech Emotion (Whisper encoder + classifier head)

Aggregates public **ungated** emotion datasets, quality-checks them, and fine-tunes an
emotion classifier on the Whisper encoder. English acted sets (RAVDESS / CREMA-D / TESS)
give volume; the small **Tamil** sets adapt + provide an honest **Tamil test** score.

**Runtime → GPU (T4)**, set `HF_USERNAME`, **Run all**. Token via Colab Secret `HF_TOKEN`.
5 classes: `angry, fear, happy, neutral, sad`.

In [ ]:
!pip -q install --upgrade "transformers>=4.44" "datasets[audio]>=2.20" "accelerate>=0.33" scikit-learn librosa soundfile huggingface_hub

In [ ]:
# ===== CONFIG =====
BACKBONE     = "openai/whisper-small"
HF_USERNAME  = "Venky0411"          # <-- your HF username
HUB_REPO     = f"{HF_USERNAME}/whisper-small-ta-emotion"
LABELS       = ["angry", "fear", "happy", "neutral", "sad"]
label2id     = {l: i for i, l in enumerate(LABELS)}
id2label     = {i: l for l, i in label2id.items()}

FREEZE_ENCODER   = True   # train only the head (fast, less overfit)
TAMIL_OVERSAMPLE = 10     # repeat the small Tamil train set to bias toward Tamil acoustics
MAX_PER_SOURCE   = None   # cap per dataset for a quick run; None = use all
EPOCHS           = 8
BATCH_SIZE       = 16
LR               = 5e-4

In [ ]:
from huggingface_hub import login
import os
HF_TOKEN = None
try:  # Colab
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    pass
if not HF_TOKEN:
    try:  # Kaggle — add a Secret named HF_TOKEN (Add-ons -> Secrets)
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass('HF write token: ')
login(token=HF_TOKEN)

In [ ]:
# ===== label normalisation + defensive loader =====
import os
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")  # avoid xet 'file size mismatch' download bug
from datasets import load_dataset, Audio, concatenate_datasets

# map every spelling/synonym to a canonical label, or None to DROP
EMO_MAP = {
    "neutral": "neutral", "calm": "neutral", "normal": "neutral",
    "happy": "happy", "happiness": "happy", "joy": "happy",
    "sad": "sad", "sadness": "sad",
    "angry": "angry", "anger": "angry", "mad": "angry",
    "fear": "fear", "fearful": "fear", "fright": "fear",
    "disgust": None, "disgusted": None,
    "surprise": None, "surprised": None, "ps": None, "pleasant surprise": None,
}
RAVDESS_CODE = {1: "neutral", 2: "calm", 3: "happy", 4: "sad",
                5: "angry", 6: "fearful", 7: "disgust", 8: "surprised"}

def canon(v):
    if v is None:
        return None
    return EMO_MAP.get(str(v).strip().lower().replace("_", " "))

def load_norm(name, source, config=None, revision=None, split="train"):
    """Load a dataset -> columns [audio(16k), emo, source]; tolerant of schema."""
    ds = load_dataset(name, config, split=split, revision=revision,
                      verification_mode="no_checks")
    cands = [c for c in ["label", "labels", "emotion", "emotions", "class"] if c in ds.column_names]
    # prefer a ClassLabel column (clean canonical names), else first candidate
    label_col = next((c for c in cands if ds.features[c].__class__.__name__ == "ClassLabel"),
                     cands[0] if cands else None)
    feat = ds.features[label_col] if label_col else None
    is_cl = feat.__class__.__name__ == "ClassLabel" if feat is not None else False
    ds = ds.cast_column("audio", Audio(decode=False))  # cheap: path/bytes only

    def _m(b):
        emo = None
        if label_col is not None:
            v = b[label_col]
            if is_cl and isinstance(v, int):
                v = feat.int2str(v)
            elif isinstance(v, int):
                v = RAVDESS_CODE.get(v, v)
            emo = canon(v)
        if emo is None:  # fallback: RAVDESS-style filename 03-01-05-...
            try:
                pth = (b["audio"] or {}).get("path") or ""
                emo = canon(RAVDESS_CODE.get(int(os.path.basename(pth).split("-")[2])))
            except Exception:
                pass
        b["emo"] = emo
        b["source"] = source
        return b

    ds = ds.map(_m)
    ds = ds.filter(lambda b: b["emo"] is not None)
    ds = ds.cast_column("audio", Audio(sampling_rate=16000))
    ds = ds.select_columns(["audio", "emo", "source"])
    if MAX_PER_SOURCE:
        ds = ds.select(range(min(MAX_PER_SOURCE, len(ds))))
    return ds

In [ ]:
# ===== load everything (skip any dataset that fails) =====
# (name, source, config, revision) — verified working sources
ENGLISH = [
    ("confit/cremad-parquet", "cremad", None, None),
    ("confit/ravdess-parquet", "ravdess", "fold1", None),  # narad/ravdess is a dead loading script
    ("AbstractTTS/TESS", "tess", None, None),
]
TAMIL = [
    # raw repo has a broken wav (audio_8.wav size mismatch); the auto-parquet branch dodges it
    ("meghana-007/tamil-audio-emotion-classification", "tamil_meghana", None, "refs/convert/parquet"),
    ("Thanushs25/tamil-audio-emotion-classification", "tamil_thanush", None, "refs/convert/parquet"),
]

def load_group(specs):
    out = []
    for name, src, cfg, rev in specs:
        try:
            d = load_norm(name, src, config=cfg, revision=rev)
            print(f"  OK  {src:16s} {len(d):6d} rows  ({name})")
            out.append(d)
        except Exception as e:
            print(f"  SKIP {src:16s} {name}: {type(e).__name__}: {str(e)[:90]}")
    return out

print("English:"); eng = load_group(ENGLISH)
print("Tamil:");   tam = load_group(TAMIL)
assert tam, "No Tamil data loaded — cannot do a Tamil eval."
english = concatenate_datasets(eng) if eng else None
tamil = concatenate_datasets(tam)

In [ ]:
# ===== QUALITY CHECK: durations + label distribution, drop bad clips =====
import numpy as np
from collections import Counter

def with_duration(ds):
    return ds.map(lambda b: {"dur": len(b["audio"]["array"]) / 16000.0}, num_proc=2)

def quality(ds, tag):
    ds = with_duration(ds)
    before = len(ds)
    ds = ds.filter(lambda b: 0.5 <= b["dur"] <= 30.0)
    durs = np.array(ds["dur"])
    print(f"[{tag}] kept {len(ds)}/{before} | dur min/mean/max = "
          f"{durs.min():.1f}/{durs.mean():.1f}/{durs.max():.1f}s")
    print(f"        labels: {dict(Counter(ds['emo']))}")
    print(f"        source: {dict(Counter(ds['source']))}")
    return ds.remove_columns(["dur"])

if english is not None:
    english = quality(english, "english")
tamil = quality(tamil, "tamil")

In [ ]:
# ===== splits: hold out a stratified Tamil TEST set; oversample Tamil TRAIN =====
from collections import Counter

tam_enc = tamil.class_encode_column("emo")   # emo -> ClassLabel (needed to stratify)
names = tam_enc.features["emo"].names
sp = tam_enc.train_test_split(test_size=0.2, seed=42, stratify_by_column="emo")

def emo_to_str(ds):
    # turn the ClassLabel emo back into a plain string column so it aligns with english
    vals = [names[i] for i in ds["emo"]]
    return ds.remove_columns("emo").add_column("emo", vals)

tamil_train = emo_to_str(sp["train"])
tamil_test  = emo_to_str(sp["test"])

train_parts = ([english] if english is not None else []) + [tamil_train] * TAMIL_OVERSAMPLE
train_ds = concatenate_datasets(train_parts).shuffle(seed=42)
print("train:", len(train_ds), "| tamil_test:", len(tamil_test))
print("train labels:", dict(Counter(train_ds["emo"])))
print("test  labels:", dict(Counter(tamil_test["emo"])))

In [ ]:
# ===== feature extraction (Whisper log-mel) + label ids =====
from transformers import WhisperFeatureExtractor
fe = WhisperFeatureExtractor.from_pretrained(BACKBONE)

def prep(b):
    a = b["audio"]
    b["input_features"] = fe(a["array"], sampling_rate=16000).input_features[0]
    b["labels"] = label2id[b["emo"]]
    return b

cols = train_ds.column_names
train_ds = train_ds.map(prep, remove_columns=cols, num_proc=2)
tamil_test = tamil_test.map(prep, remove_columns=tamil_test.column_names, num_proc=2)

In [ ]:
# ===== model: Whisper encoder + classification head =====
from transformers import WhisperForAudioClassification
model = WhisperForAudioClassification.from_pretrained(
    BACKBONE, num_labels=len(LABELS), label2id=label2id, id2label=id2label)
if FREEZE_ENCODER:
    for p in model.encoder.parameters():
        p.requires_grad = False
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable/1e6:.2f}M")

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

In [ ]:
from transformers import (TrainingArguments, Trainer, default_data_collator)

args = TrainingArguments(
    output_dir="./whisper-ta-emotion",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    warmup_ratio=0.1,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to=["none"],
    push_to_hub=True,
    hub_model_id=HUB_REPO,
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=tamil_test,
    data_collator=default_data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# ===== honest Tamil-test report =====
from sklearn.metrics import classification_report, confusion_matrix
pred = trainer.predict(tamil_test)
y, yhat = pred.label_ids, pred.predictions.argmax(1)
print(classification_report(y, yhat, target_names=LABELS, digits=3))
print("confusion matrix (rows=true):\n", confusion_matrix(y, yhat))

In [ ]:
# ===== push model + feature extractor =====
fe.push_to_hub(HUB_REPO)
trainer.push_to_hub(**{"dataset": "RAVDESS+CREMA-D+TESS+Tamil", "language": "ta",
                       "tasks": "audio-classification", "finetuned_from": BACKBONE})
print("Pushed:", HUB_REPO)

In [ ]:
# ===== quick inference demo =====
import torch, librosa
def predict_emotion(path):
    y, _ = librosa.load(path, sr=16000, mono=True)
    feats = fe(y, sampling_rate=16000, return_tensors="pt").input_features.to(model.device)
    with torch.no_grad():
        probs = model(feats).logits.softmax(-1)[0]
    i = int(probs.argmax())
    return id2label[i], float(probs[i])

# example: predict_emotion('/content/sample.wav')